Analyzes yearly LST and NDVI differences between new housing development areas and reference areas before and after construction start, including temporal trajectories, significance tests, and percentile-limited visualizations.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
import re
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# ============================================
# CONFIGURATION
# ============================================
print("=" * 80)
print("NDVI/LST BEFORE VS AFTER CONSTRUCTION YEAR")
print("=" * 80)

COMBINED_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB.gpkg"
COMBINED_LAYER = "Comparison_LST_NDVI_DEGURB"
CONSTRUCTION_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

OUTPUT_DIR             = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Pre_Post_Construction"
MIN_OBS_FOR_TRAJECTORY = 5   # minimum n to appear in trajectory plot
MIN_N_FOR_TEST         = 10  # minimum n to run a significance test


def build_difference_long(frame, metric_name):
    """Convert wide yearly difference columns into a long table."""
    pattern = re.compile(rf"^difference_{metric_name}_(\d{{4}})$")
    year_cols = []
    year_lookup = {}

    for column in frame.columns:
        match = pattern.match(column)
        if match:
            year_cols.append(column)
            year_lookup[column] = int(match.group(1))

    if not year_cols:
        raise ValueError(f"No columns found for metric {metric_name}. Expected columns like difference_{metric_name}_2019")

    long_df = frame[['nhda_id', *year_cols]].melt(
        id_vars='nhda_id',
        value_vars=year_cols,
        var_name='source_column',
        value_name=f'{metric_name}_diff',
    )
    long_df['year'] = long_df['source_column'].map(year_lookup)
    long_df = long_df.drop(columns=['source_column'])
    long_df = long_df.dropna(subset=[f'{metric_name}_diff']).copy()
    long_df['year'] = long_df['year'].astype(int)

    return long_df

# ============================================
# 1. LOAD INPUT DATA
# ============================================
print("\n1. Loading inputs...")

for path in [COMBINED_GPKG, CONSTRUCTION_GPKG]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required input not found: {path}")

gdf_metrics = gpd.read_file(COMBINED_GPKG, layer=COMBINED_LAYER)
gdf_const = gpd.read_file(CONSTRUCTION_GPKG)

if 'nhda_id' not in gdf_metrics.columns:
    raise ValueError("Combined GPKG is missing column: nhda_id")

gdf_metrics['nhda_id'] = gdf_metrics['nhda_id'].astype(str)

# Filter to NHDA only to avoid duplicates
row_type_col = None
for candidate in ['type', 'area_type']:
    if candidate in gdf_metrics.columns:
        row_type_col = candidate
        break


if row_type_col is not None and 'NHDA' in set(gdf_metrics[row_type_col].dropna().astype(str)):
    metrics_base = gdf_metrics[gdf_metrics[row_type_col].astype(str) == 'NHDA'].copy()
else:
    metrics_base = gdf_metrics.drop_duplicates(subset=['nhda_id']).copy()

metrics_base = metrics_base.drop_duplicates(subset=['nhda_id']).copy()

lst_long = build_difference_long(metrics_base, 'LST')
ndvi_long = build_difference_long(metrics_base, 'NDVI')
metrics = pd.merge(lst_long, ndvi_long, on=['nhda_id', 'year'], how='outer')
metrics['nhda_id'] = metrics['nhda_id'].astype(str)
metrics['year'] = pd.to_numeric(metrics['year'], errors='coerce')
metrics = metrics.dropna(subset=['year']).copy()
metrics['year'] = metrics['year'].astype(int)

print(f"   Input rows in combined GPKG: {len(gdf_metrics)}")
print(f"   Rows used for yearly metrics: {len(metrics_base)}")
print(f"   LST observations:            {len(lst_long)}")
print(f"   NDVI observations:           {len(ndvi_long)}")
print(f"   Merged rows:                 {len(metrics)}")

const_cols = list(gdf_const.columns)
id_candidates = ['nhda_id', 'cluster_id', 'nda_id']
construction_candidates = ['construction_start', 'construction_year', 'construction_start_year']

id_col = next((column for column in id_candidates if column in const_cols), None)
construction_col = next((column for column in construction_candidates if column in const_cols), None)

if id_col is None:
    raise ValueError(f"No ID column found in construction GPKG. Tried: {id_candidates}")
if construction_col is None:
    raise ValueError(f"No construction year column found in construction GPKG. Tried: {construction_candidates}")

const_df = gdf_const[[id_col, construction_col]].copy()
const_df['nhda_id'] = const_df[id_col].astype(str)
construction_raw = const_df[construction_col].astype(str).str.strip()
construction_special_map = {
    'AUC_2015': 2014,
    'AUC_2016': 2015,
}
const_df['construction_year'] = construction_raw.replace(construction_special_map)
const_df['construction_year'] = pd.to_numeric(const_df['construction_year'], errors='coerce')
const_df = const_df[['nhda_id', 'construction_year']].drop_duplicates(subset=['nhda_id'])

print(f"   Construction records:          {len(const_df)}")
print(f"   Construction year source col:  {construction_col}")
mapped_count = construction_raw.isin(construction_special_map).sum()
print(f"   Special already-under-construction values mapped: {mapped_count}")

# ============================================
# 2. BUILD ANALYSIS DATASET
# ============================================
print("\n2. Building analysis dataset...")

df = pd.merge(metrics, const_df, on='nhda_id', how='left')
df = df[df['construction_year'].notna()].copy()

if df.empty:
    raise ValueError("No records left after joining construction years.")

df['years_since_start'] = df['year'] - df['construction_year']
df['phase'] = np.where(df['year'] < df['construction_year'], 'Pre', 'Post')
df = df[~(df['LST_diff'].isna() & df['NDVI_diff'].isna())].copy()

print(f"   Final observations: {len(df)}")
print(f"   Unique NHDAs:       {df['nhda_id'].nunique()}")
print(f"   Year range:         {df['year'].min()} - {df['year'].max()}")

# ============================================
# 3. SUMMARY STATISTICS
# ============================================
print("\n3. Calculating summaries...")

trajectory = df.groupby('years_since_start').agg(
    LST_count=('LST_diff', 'count'),
    LST_mean=('LST_diff', 'mean'),
    LST_std=('LST_diff', 'std'),
    LST_median=('LST_diff', 'median'),
    NDVI_mean=('NDVI_diff', 'mean'),
    NDVI_std=('NDVI_diff', 'std'),
    NDVI_median=('NDVI_diff', 'median'),
).round(4)

phase_stats = df.groupby('phase').agg(
    LST_count=('LST_diff', 'count'),
    LST_mean=('LST_diff', 'mean'),
    LST_std=('LST_diff', 'std'),
    LST_median=('LST_diff', 'median'),
    NDVI_mean=('NDVI_diff', 'mean'),
    NDVI_std=('NDVI_diff', 'std'),
    NDVI_median=('NDVI_diff', 'median'),
).round(4)

print("\n" + "=" * 80)
print("PHASE COMPARISON (PRE VS POST)")
print("=" * 80)
print(phase_stats)

if 'Pre' in phase_stats.index and 'Post' in phase_stats.index:
    print("\n--- Changes (Post - Pre) ---")
    print(f"LST:  {phase_stats.loc['Post', 'LST_mean'] - phase_stats.loc['Pre', 'LST_mean']:+.3f} °C")
    print(f"NDVI: {phase_stats.loc['Post', 'NDVI_mean'] - phase_stats.loc['Pre', 'NDVI_mean']:+.4f}")

# ============================================
# 3b. SIGNIFICANCE TESTING — PER YEAR SINCE CONSTRUCTION
# ============================================
print("\n" + "=" * 80)
print("SIGNIFICANCE TESTING - PER YEAR SINCE CONSTRUCTION START")
print(f"(only years with n >= {MIN_N_FOR_TEST} are tested; FDR correction applied across tested years)")
print("=" * 80)

def interpret_r(r):
    r = abs(r)
    if r < 0.1:
        return "negligible"
    if r < 0.3:
        return "small"
    if r < 0.5:
        return "medium"
    return "large"

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_year_results = []

for metric, label, unit in [
    ('LST_diff', 'LST', '°C'),
    ('NDVI_diff', 'NDVI', ''),
]:
    print(f"\n--- {label} ---")
    year_rows = []

    for year_since_start, group in df.groupby('years_since_start'):
        values = group[metric].dropna().values
        count = len(values)

        if count < MIN_N_FOR_TEST:
            year_rows.append({
                'years_since_start': year_since_start,
                'n': count,
                'median': round(np.median(values), 4) if count > 0 else np.nan,
                'mean': round(np.mean(values), 4) if count > 0 else np.nan,
                'std': round(np.std(values), 4) if count > 0 else np.nan,
                'p_raw': np.nan,
                'p_fdr': np.nan,
                'W': np.nan,
                'r': np.nan,
                'effect': 'n/a',
                'tested': False,
                'significant_fdr': False,
                'metric': label,
            })
            continue

        try:
            statistic, p_value = stats.wilcoxon(values, alternative='two-sided')
        except ValueError:
            year_rows.append({
                'years_since_start': year_since_start,
                'n': count,
                'median': round(np.median(values), 4),
                'mean': round(np.mean(values), 4),
                'std': round(np.std(values), 4),
                'p_raw': np.nan,
                'p_fdr': np.nan,
                'W': np.nan,
                'r': np.nan,
                'effect': 'n/a',
                'tested': False,
                'significant_fdr': False,
                'metric': label,
            })
            continue

        mu_w = count * (count + 1) / 4
        sd_w = np.sqrt(count * (count + 1) * (2 * count + 1) / 24)
        z_score = (statistic - mu_w) / sd_w
        effect_size = abs(z_score) / np.sqrt(count)

        year_rows.append({
            'years_since_start': year_since_start,
            'n': count,
            'median': round(np.median(values), 4),
            'mean': round(np.mean(values), 4),
            'std': round(np.std(values), 4),
            'p_raw': round(p_value, 6),
            'p_fdr': np.nan,
            'W': round(statistic, 2),
            'r': round(effect_size, 3),
            'effect': interpret_r(effect_size),
            'tested': True,
            'significant_fdr': False,
            'metric': label,
        })

    yearly_results = pd.DataFrame(year_rows).sort_values('years_since_start').reset_index(drop=True)
    tested_mask = yearly_results['tested']

    if tested_mask.sum() > 0:
        p_values = yearly_results.loc[tested_mask, 'p_raw'].values
        reject, p_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
        yearly_results.loc[tested_mask, 'p_fdr'] = p_fdr.round(6)
        yearly_results.loc[tested_mask, 'significant_fdr'] = reject

    all_year_results.append(yearly_results)

    print(f"\n   {'Year':>6} {'n':>5} {'Median':>8} {'Mean':>8} {'W':>10} {'p_raw':>8} {'p_fdr':>8} {'r':>6} {'Effect':<12} {'Sig(FDR)':>8} {'Tested':>12}")
    print("   " + "-" * 100)

    for _, row in yearly_results.iterrows():
        tested_str = "yes" if row['tested'] else f"no (n<{MIN_N_FOR_TEST})"
        sig_str = "yes" if row['significant_fdr'] else (
        "no" if row['tested'] else "-")
        p_raw_str = f"{row['p_raw']:.4f}" if not np.isnan(row['p_raw']) else "-"
        p_fdr_str = f"{row['p_fdr']:.4f}" if not np.isnan(row['p_fdr']) else "-"
        r_str = f"{row['r']:.3f}" if not np.isnan(row['r']) else "-"
        w_str = f"{row['W']:.1f}" if not np.isnan(row['W']) else "-"
        print(
            f"   {int(row['years_since_start']):>6} {int(row['n']):>5} {row['median']:>8.4f} "
            f"{row['mean']:>8.4f} {w_str:>10} {p_raw_str:>8} {p_fdr_str:>8} "
            f"{r_str:>6} {row['effect']:<12} {sig_str:>8} {tested_str:>12}"
        )

print("\n\n" + "=" * 80)
print("OVERALL PRE VS POST - Wilcoxon signed-rank on per-NHDA means")
print("=" * 80)

overall_results = []
for metric, label, unit in [('LST_diff', 'LST', '°C'), ('NDVI_diff', 'NDVI', '')]:
    nhda_means = (
        df[df[metric].notna()]
        .groupby(['nhda_id', 'phase'])[metric]
        .mean()
        .unstack('phase')
    )
    paired = nhda_means.dropna(subset=['Pre', 'Post'])
    paired_count = len(paired)
    print(f"\n   {label}: {paired_count} NHDAs with both Pre and Post data")

    if paired_count >= MIN_N_FOR_TEST:
        statistic, p_value = stats.wilcoxon(paired['Pre'].values, paired['Post'].values, alternative='two-sided')
        mu_w = paired_count * (paired_count + 1) / 4
        sd_w = np.sqrt(paired_count * (paired_count + 1) * (2 * paired_count + 1) / 24)
        z_score = (statistic - mu_w) / sd_w
        effect_size = abs(z_score) / np.sqrt(paired_count)
        significance = "significant" if p_value < 0.05 else "not significant"
        print(f"   W={statistic:.1f}, p={p_value:.4f} -> {significance}")
        print(f"   Effect size r={effect_size:.3f} ({interpret_r(effect_size)})")
        overall_results.append({
            'metric': label,
            'n_paired': paired_count,
            'W': round(statistic, 1),
            'p': round(p_value, 6),
            'r': round(effect_size, 3),
            'effect': interpret_r(effect_size),
        })
    else:
        print(f"   Skipped - fewer than {MIN_N_FOR_TEST} paired NHDAs.")

# ============================================
# 4. VISUALIZATIONS
# ============================================
print("\n4. Creating plots...")

traj_plot = trajectory[trajectory['LST_count'] >= MIN_OBS_FOR_TRAJECTORY].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x_values = traj_plot.index.values

for ax, mean_col, std_col, color, ylabel, title in [
    (axes[0], 'LST_mean', 'LST_std', 'steelblue', 'LST Difference (NHDA - RA) [°C]', 'LST Difference around Construction Year'),
    (axes[1], 'NDVI_mean', 'NDVI_std', 'green', 'NDVI Difference (NHDA - RA)', 'NDVI Difference around Construction Year'),
]:
    ax.errorbar(
        x_values,
        traj_plot[mean_col].values,
        yerr=traj_plot[std_col].values,
        marker='o',
        capsize=4,
        linewidth=2,
        color=color,
        label='Mean ± SD',
    )
    ax.axhline(0, linestyle='--', color='red', linewidth=1.8)
    ax.axvline(0, linestyle=':', color='gray', linewidth=1.8, label='Construction year')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Years since construction year')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pre_post_trajectory_median.png", dpi=300, bbox_inches='tight')
print("   Saved: pre_post_trajectory.png")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, label, unit, palette in [
    (axes[0], 'LST_diff', 'LST', '°C', 'Blues'),
    (axes[1], 'NDVI_diff', 'NDVI', '', 'Greens'),
]:
    sns.boxplot(
        data=df,
        x='phase',
        y=metric,
        order=['Pre', 'Post'],
        ax=ax,
        palette=palette
    )

    # Limit y-axis to the 1st–99th percentile
    lower = df[metric].quantile(0.01)
    upper = df[metric].quantile(0.99)
    ax.set_ylim(lower, upper)

    ax.axhline(0, linestyle='--', color='red', linewidth=1.5)
    ax.set_title(f'{label} Difference: Pre vs Post', fontweight='bold')
    ax.set_xlabel('Phase')
    ax.set_ylabel(f'{label} Difference (NHDA - RA) {unit}'.strip())
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pre_post_boxplots_median.png", dpi=300, bbox_inches='tight')
print("   Saved: pre_post_boxplots.png")
plt.close()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
# fig.suptitle(
#     'Significance of NHDA vs. Reference Area Difference per Year since Construction Start',
#     fontweight='bold',
#     fontsize=13,
# )

ALPHA_FDR = 0.05
COLOR_ZERO = 'red'
COLOR_SKIP = '#cccccc'

# Prepare data for boxplots
for ax_idx, (ax, yearly_results, label, unit, metric_color) in enumerate([
    (axes[0], all_year_results[0], 'LST', '°C', '#a4133c'),
    (axes[1], all_year_results[1], 'NDVI', '', '#2d6a4f'),
]):
    # Collect data for boxplot by year_since_start
    box_data = []
    positions = []
    colors = []
    x_ticks = []
    x_tick_labels = []
    
    for year_val in sorted(yearly_results['years_since_start'].unique()):
        year_subset = df[df['years_since_start'] == year_val]
        
        if label == 'LST':
            metric_col = 'LST_diff'
        else:
            metric_col = 'NDVI_diff'
        
        metric_vals = year_subset[metric_col].dropna().values
        
        if len(metric_vals) > 0:
            box_data.append(metric_vals)
            positions.append(year_val)
            
            # Determine color based on significance
            year_row = yearly_results[yearly_results['years_since_start'] == year_val].iloc[0]
            if not year_row['tested']:
                colors.append(COLOR_SKIP)
            elif year_row['significant_fdr']:
                colors.append(metric_color)
            else:
                # Non-significant: use light gray
                colors.append('#bfbfbf')
            
            x_ticks.append(year_val)
            x_tick_labels.append(str(int(year_val)))
    
    # Draw boxplots
    bp = ax.boxplot(
        box_data,
        positions=positions,
        widths=0.6,
        patch_artist=True,
        medianprops=dict(color='darkred', linewidth=1.5),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        boxprops=dict(linewidth=1.2),
        flierprops=dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5),
    )
    
    # Color each box
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add sample size labels above boxes
    for pos, year_val in zip(positions, x_ticks):
        year_row = yearly_results[yearly_results['years_since_start'] == year_val].iloc[0]
        n_val = int(year_row['n'])
        ax.text(
            pos, ax.get_ylim()[1] * 0.95,
            f'n={n_val}',
            ha='center', va='top',
            fontsize=10,
            color = "#6E6D6D",
            bbox=dict(facecolor='white', edgecolor='none', pad=1.5)
        )
    
    # Reference lines
    ax.axhline(0, linestyle='--', color=COLOR_ZERO, linewidth=1.5, zorder=3, alpha=0.7)
    ax.axvline(0, linestyle='--', color="#bbbd71", linewidth=1.8, alpha=1)
    
    # Labels and formatting
    ax.set_ylabel(
        f'{label} Difference (NHDA - RA) {unit}'.strip(),
        fontsize=12, fontweight='bold'
    )
    ax.set_title(
        f'{label}: Difference Distribution per Year since Construction Start',
        fontweight='bold', fontsize=12, pad=10
    )
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels, fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(True, axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[1].set_xlabel('Years since construction start', fontsize=12, fontweight='bold')

# Create legend outside the plots
legend_elements = [
    mpatches.Patch(facecolor='#a4133c', alpha=0.7, label='LST (significant)'),
    mpatches.Patch(facecolor='#2d6a4f', alpha=0.7, label='NDVI (significant)'),
    mpatches.Patch(facecolor='#bfbfbf', alpha=0.7, label='Not significant'),
    plt.Line2D([0], [0], color=COLOR_ZERO, linestyle='--', linewidth=1.5, label='No difference (NHDA = RA)'),
    plt.Line2D([0], [0], color="#bbbd71", linestyle='--', linewidth=1.8, label='Construction start'),
]

fig.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=5,
    fontsize=11,
    frameon=False,
    borderpad=0
)

plt.tight_layout(rect=[0, 0.05, 1, 0.98])
plt.savefig(f"{OUTPUT_DIR}/significance_per_year_median.png", dpi=300, bbox_inches='tight')
print("   Saved: significance_per_year_median.png")
plt.close()

fig, axes = plt.subplots(
    3,
    1,
    figsize=(18, 13),
    gridspec_kw={'height_ratios': [2, 2, 1]},
    sharex=True,
)
fig.suptitle(
    'p-value Timeline and Sample Size per Year since Construction Start',
    fontweight='bold',
    fontsize=13,
)

for ax, yearly_results, label, color in [
    (axes[0], all_year_results[0], 'LST', 'steelblue'),
    (axes[1], all_year_results[1], 'NDVI', 'green'),
]:
    tested_results = yearly_results[yearly_results['tested']].sort_values('years_since_start')
    skipped_results = yearly_results[~yearly_results['tested']].sort_values('years_since_start')

    x_tested = tested_results['years_since_start'].values
    p_tested = tested_results['p_fdr'].values
    ax.plot(x_tested, p_tested, color=color, linewidth=1.5, alpha=0.6, zorder=1)

    significant_mask = tested_results['significant_fdr'].values
    ax.scatter(x_tested[significant_mask], p_tested[significant_mask], color='#2ecc71', s=70, zorder=3, label='Significant')
    ax.scatter(x_tested[~significant_mask], p_tested[~significant_mask], color='#e74c3c', s=70, zorder=3, label='Not significant')

    for x_pos in skipped_results['years_since_start'].values:
        ax.axvline(x_pos, color='#bdc3c7', linewidth=0.8, alpha=0.5, linestyle=':')

    ax.axhline(ALPHA_FDR, color='black', linestyle='--', linewidth=1.3, label=f'alpha = {ALPHA_FDR}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=1.5)
    ax.set_ylabel(f'{label}\nFDR-corrected p', fontsize=10)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.25, axis='y')
    ax.legend(fontsize=9, loc='upper right', framealpha=0.85)
    ax.set_title(f'{label} - FDR-corrected p-value per Year', fontweight='bold')

year_reference = all_year_results[0].sort_values('years_since_start')
bar_colors = [
    '#2ecc71' if row['significant_fdr'] else ('#e74c3c' if row['tested'] else '#bdc3c7')
    for _, row in year_reference.iterrows()
]

axes[2].bar(
    year_reference['years_since_start'].values,
    year_reference['n'].values,
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
)
axes[2].axvline(0, color='gray', linestyle=':', linewidth=1.5)
axes[2].axhline(MIN_N_FOR_TEST, color='black', linestyle='--', linewidth=1.2, label=f'Min. n = {MIN_N_FOR_TEST}')

for _, row in year_reference.iterrows():
    axes[2].text(
        row['years_since_start'],
        row['n'] + 0.3,
        str(int(row['n'])),
        ha='center',
        va='bottom',
        fontsize=7,
    )

axes[2].set_ylabel('n (observations)', fontsize=10)
axes[2].set_xlabel('Years since construction start', fontsize=11)
axes[2].set_title('Sample Size per Year', fontweight='bold')
axes[2].legend(fontsize=9, loc='upper right')
axes[2].grid(alpha=0.25, axis='y')
axes[2].set_xticks(sorted(year_reference['years_since_start'].unique()))
axes[2].tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pvalue_timeline.png", dpi=300, bbox_inches='tight')
print("   Saved: pvalue_timeline.png")
plt.close()

# ============================================
# 5. EXPORT TABLES
# ============================================
print("\n5. Saving tables...")

trajectory.to_csv(f"{OUTPUT_DIR}/trajectory_by_years_pre_post.csv")
phase_stats.to_csv(f"{OUTPUT_DIR}/phase_comparison_pre_post.csv")
df.to_csv(f"{OUTPUT_DIR}/all_observations_pre_post_median.csv", index=False)

combined_sig = pd.concat(all_year_results, ignore_index=True)
combined_sig.to_csv(f"{OUTPUT_DIR}/significance_per_year.csv", index=False)

if overall_results:
    pd.DataFrame(overall_results).to_csv(f"{OUTPUT_DIR}/significance_overall_prepost_median.csv", index=False)

print("   Saved: trajectory_by_years_pre_post.csv")
print("   Saved: phase_comparison_pre_post.csv")
print("   Saved: all_observations_pre_post.csv")
print("   Saved: significance_per_year.csv")
print("   Saved: significance_overall_prepost.csv")

print("\n" + "=" * 80)
print("PRE/POST ANALYSIS COMPLETE")
print("=" * 80)